# Building e-commerce dataset from WILDBERRIES reviews

In [ ]:
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent
os.chdir(BASE_DIR)


## Loading real WB reviews

In [ ]:
def normalize_text(text: str) -> str:
    text = str(text or '').replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_real_wb_reviews(limit=1500, min_chars=40, max_chars=450):
    ds_wb = load_dataset('nyuuzyou/wb-feedbacks', split='train', streaming=True)
    rows, seen = [], set()

    for item in ds_wb:
        text = normalize_text(item.get('text'))
        if not (min_chars <= len(text) <= max_chars):
            continue
        if text.lower() in seen:
            continue
        seen.add(text.lower())

        rows.append({
            'real_id': f'wb_real_{len(rows):05d}',
            'text': text,
        })
        if len(rows) >= limit:
            break

    return pd.DataFrame(rows)

real_wb_df = load_real_wb_reviews(limit=1500)
display(real_wb_df.head(3))

## Monitoring of Surface Signals

In [ ]:
POSITIVE_MARKERS = [
    'отлично', 'отличный', 'отличная', 'супер', 'класс', 'понравилось',
    'понравился', 'доволен', 'довольна', 'хорошо', 'нравится', 'удобно'
]

NEGATIVE_MARKERS = [
    ' но ', 'однако', 'зато', 'хотя', 'жаль', 'к сожалению', 'минус',
    'недостаток', 'плохо', 'не понрав', 'разочар', 'брак', 'дефект',
    'сломал', 'порвал', 'не подош', 'маловат', 'великоват', 'возврат'
]

TEMPLATE_PHRASES = [
    'всем рекомендую', 'однозначно рекомендую', 'не пожалеете',
    'качество на высоте', 'продавцу спасибо', 'буду брать еще',
    'буду заказывать еще', 'в подарок', 'идеальный товар',
    'выше всяких похвал', 'на все сто', 'покупкой довольна',
    'покупкой доволен'
]


def count_markers(text, markers):
    t = f' {normalize_text(text).lower()} '
    return sum(1 for marker in markers if marker in t)


def sentiment_bucket(text):
    pos = count_markers(text, POSITIVE_MARKERS)
    neg = count_markers(text, NEGATIVE_MARKERS)
    if pos and neg:
        return 'mixed'
    if neg:
        return 'negative'
    if pos:
        return 'positive'
    return 'neutral'


def pair_audit(real_text, fake_text):
    real_text = normalize_text(real_text)
    fake_text = normalize_text(fake_text)
    ratio = len(fake_text) / max(len(real_text), 1)
    return {
        'real_len': len(real_text),
        'fake_len': len(fake_text),
        'len_ratio': ratio,
        'real_bucket': sentiment_bucket(real_text),
        'fake_bucket': sentiment_bucket(fake_text),
        'real_template_count': count_markers(real_text, TEMPLATE_PHRASES),
        'fake_template_count': count_markers(fake_text, TEMPLATE_PHRASES),
        'real_negative_count': count_markers(real_text, NEGATIVE_MARKERS),
        'fake_negative_count': count_markers(fake_text, NEGATIVE_MARKERS),
        'real_positive_count': count_markers(real_text, POSITIVE_MARKERS),
        'fake_positive_count': count_markers(fake_text, POSITIVE_MARKERS),
    }


def is_quality_pair(real_text, fake_text):
    audit = pair_audit(real_text, fake_text)
    length_ok = 0.75 <= audit['len_ratio'] <= 1.30
    template_ok = audit['fake_template_count'] <= 1
    sentiment_ok = not (
        audit['real_bucket'] in {'negative', 'mixed'}
        and audit['fake_bucket'] == 'positive'
        and audit['fake_negative_count'] == 0
    )
    return length_ok and template_ok and sentiment_ok


## Generating Paired Fake Reviews

In [ ]:
FAKE_INTENTS = [
    'нейтрально имитировать обычный покупательский отзыв',
    'продвинуть товар без явной рекламы',
    'защитить товар после возможной жалобы',
    'преувеличить достоинства, сохранив бытовой стиль',
    'замаскировать небольшой недостаток товара',
    'атаковать товар через правдоподобный негативный отзыв',
]

PROMPT_PAIRED = """
Ты создаёшь данные для исследования методов NLP по выявлению fake/заказных отзывов в e-commerce.

Ниже дан реальный отзыв покупателя Wildberries:
"{real_review}"

Интент искусственного отзыва: {intent}.

Напиши искусственно подготовленный fake-отзыв про тот же товар.
Fake-отзыв должен быть похож на обычный WB-отзыв, а не на рекламный текст.

Правила:
1. Сохрани товарную ситуацию и основные детали использования.
2. Не превращай негативный отзыв автоматически в восторженную рекламу.
3. Если интент "атаковать товар", fake может быть более негативным, но всё равно должен звучать как реальный опыт покупателя.
4. Если интент "защитить" или "продвинуть", не делай отзыв слишком гладким: оставь бытовую деталь, сомнение или мелкий минус.
5. Длина fake должна быть почти такой же, как у real: желательно 85-105% от длины исходного текста, максимум 110%.
6. Не добавляй подарок, бонус, благодарность продавцу, если этого не было в исходнике.
7. Не используй шаблонные фразы: "всем рекомендую", "не пожалеете", "качество на высоте", "просто супер", "буду брать ещё".
8. Пиши живым русским языком WB-отзыва: допускаются разговорность, небольшие пунктуационные ошибки, короткие фразы.
9. Не копируй исходник дословно: измени формулировки, но сохрани тему.

Few-shot примеры нужного стиля:
Real: Пришло быстро, но упаковка была помята. Сам товар целый, пользоваться можно.
Fake: Доставили без задержки, коробка немного пострадала, но внутри всё нормально. В работе пока устраивает.

Real: Размер подошёл, ткань тонкая, после стирки форму не потеряла.
Fake: По размеру всё совпало, материал не плотный. Постирала один раз, вид остался нормальный.

Real: Крем пришёл целый, запах спокойный, впитывается нормально.
Fake: Упаковка была целая, но сам крем не очень зашёл. Запах странноватый, впитывается долго, второй раз вряд ли возьму.

Самопроверка перед ответом:
- fake не длиннее real больше чем на 10%;
- нет явной рекламы и шаблонных рекомендаций;
- если real негативный, fake не стал резко восторженным без причины;
- среди fake встречаются разные интенты: продвинуть, защитить, преувеличить, атаковать.
Если fake получился длиннее, сократи его до 1-3 предложений.

Верни только текст fake-отзыва. Без пояснений, кавычек и JSON.
""".strip()

GENERATOR_MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-5')
PAIRED_PATH = BASE_DIR / 'paired_reviews_len_balanced.json'
MAX_FAKE_REAL_RATIO = 1.05
MIN_USEFUL_RATIO = 0.65


def get_anthropic_client():
    api_key = os.getenv('ANTHROPIC_API_KEY')
    if not api_key:
        from getpass import getpass
        api_key = getpass('enter api key').strip()
        if not api_key:
            raise RuntimeError('ANTHROPIC_API_KEY was not find')
        os.environ['ANTHROPIC_API_KEY'] = api_key
    import anthropic
    return anthropic.Anthropic(api_key=api_key)


def trim_fake_to_real_length(real_text, fake_text, max_ratio=MAX_FAKE_REAL_RATIO):
    real_text = normalize_text(real_text)
    fake_text = normalize_text(fake_text)
    max_chars = max(35, int(len(real_text) * max_ratio))
    if len(fake_text) <= max_chars:
        return fake_text

    sentences = [s.strip() for s in re.split(r'(?<=[.!?…])\s+', fake_text) if s.strip()]
    kept = []
    for sentence in sentences:
        candidate = normalize_text(' '.join(kept + [sentence]))
        if len(candidate) <= max_chars:
            kept.append(sentence)
        else:
            break

    by_sentence = normalize_text(' '.join(kept))
    if len(by_sentence) >= max(30, int(len(real_text) * MIN_USEFUL_RATIO)):
        return by_sentence

    hard = fake_text[:max_chars].rsplit(' ', 1)[0].strip()
    hard = re.sub(r'[,;:\-]+$', '', hard).strip()
    if hard and hard[-1] not in '.!?…':
        hard += '.'
    return normalize_text(hard)


def generate_fake_review(client, real_text, intent=None, max_tokens=320):
    if intent is None:
        intent = random.choice(FAKE_INTENTS)

    resp = client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=max_tokens,
        temperature=0.9,
        messages=[{
            'role': 'user',
            'content': PROMPT_PAIRED.format(real_review=real_text, intent=intent)
        }]
    )
    text = normalize_text(resp.content[0].text)
    text = re.sub(r'^(fake|ответ)\s*[:\-]\s*', '', text, flags=re.IGNORECASE).strip(' "«»')
    return trim_fake_to_real_length(real_text, text)

In [ ]:
GENERATE_NEW_PAIRS = False
TARGET_PAIRS = 500
MAX_ATTEMPTS_PER_REVIEW = 3
SLEEP_SECONDS = 0.25

if GENERATE_NEW_PAIRS:
    client = get_anthropic_client()
    paired_data = []
    failed = 0

    sample_reals = real_wb_df.sample(
        n=min(TARGET_PAIRS, len(real_wb_df)),
        random_state=SEED
    ).reset_index(drop=True)

    for i, row in sample_reals.iterrows():
        real_text = row['text']
        accepted_fake = None
        accepted_intent = None

        for attempt in range(1, MAX_ATTEMPTS_PER_REVIEW + 1):
            try:
                intent = random.choice(FAKE_INTENTS)
                fake_text = generate_fake_review(client, real_text, intent=intent)
                audit = pair_audit(real_text, fake_text)
                length_ok = 0.45 <= audit['len_ratio'] <= 1.10
                template_ok = audit['fake_template_count'] <= 3
                if fake_text != real_text and len(fake_text) >= 30 and length_ok and template_ok:
                    accepted_fake = fake_text
                    accepted_intent = intent
                    break
            except Exception as e:
                failed += 1
                break
            time.sleep(SLEEP_SECONDS)

        if accepted_fake:
            paired_data.append({
                'pair_id': f'wb_pair_{len(paired_data):05d}',
                'real': real_text,
                'fake': accepted_fake,
                'real_id': row.get('real_id', ''),
                'generator_model': GENERATOR_MODEL,
                'prompt_version': 'claude_intents_len_balanced_500',
                'fake_intent': accepted_intent,
                **pair_audit(real_text, accepted_fake),
            })


    with open(PAIRED_PATH, 'w', encoding='utf-8') as f:
        json.dump(paired_data, f, ensure_ascii=False, indent=2)

else:
    with open(PAIRED_PATH, encoding='utf-8') as f:
        paired_data = json.load(f)


In [ ]:
RESUME_TARGET_PAIRS = 500
MAX_REVIEWS_TO_TRY = 1000
SLEEP_SECONDS = 0.15

PAIRED_PATH = Path('paired_reviews_len_balanced.json')

with open(PAIRED_PATH, encoding='utf-8') as f:
    paired_data = json.load(f)

used_texts = {
    normalize_text(p.get('real', '')).lower()
    for p in paired_data
}

ood_path = BASE_DIR / 'data' / 'ood_wb_50.csv'
if ood_path.exists():
    ood_df_for_exclusion = pd.read_csv(ood_path)
    ood_texts = {
        normalize_text(t).lower()
        for t in ood_df_for_exclusion['text'].dropna().tolist()
    }
    used_texts |= ood_texts

candidate_reals = real_wb_df[
    ~real_wb_df['text'].map(lambda x: normalize_text(x).lower()).isin(used_texts)
].copy()

candidate_reals = candidate_reals.sample(
    n=min(MAX_REVIEWS_TO_TRY, len(candidate_reals)),
    random_state=SEED + len(paired_data)
).reset_index(drop=True)

client = get_anthropic_client()
start_count = len(paired_data)
accepted_by_intent = {intent: 0 for intent in FAKE_INTENTS}

for i, row in candidate_reals.iterrows():
    if len(paired_data) >= RESUME_TARGET_PAIRS:
        break

    real_text = row['text']

    try:
        intent = FAKE_INTENTS[(len(paired_data) + i) % len(FAKE_INTENTS)]
        fake_text = generate_fake_review(client, real_text, intent=intent)
        audit = pair_audit(real_text, fake_text)

        length_ok = 0.45 <= audit['len_ratio'] <= 1.10
        template_ok = audit['fake_template_count'] <= 3
        not_duplicate = fake_text != real_text
        not_too_short = len(fake_text) >= 30

        if not_duplicate and not_too_short and length_ok and template_ok:
            paired_data.append({
                'pair_id': f'wb_pair_{len(paired_data):05d}',
                'orig_pair_id': row.get('pair_id', ''),
                'real': real_text,
                'fake': fake_text,
                'real_id': row.get('real_id', ''),
                'generator_model': GENERATOR_MODEL,
                'prompt_version': 'claude_intents_len_balanced_500',
                'fake_intent': intent,
                'quality_ok': True,
                **audit,
            })
            accepted_by_intent[intent] += 1

    except Exception as e:
        break

    if len(paired_data) > start_count and len(paired_data) % 10 == 0:
        with open(PAIRED_PATH, 'w', encoding='utf-8') as f:
            json.dump(paired_data, f, ensure_ascii=False, indent=2)

    time.sleep(SLEEP_SECONDS)

with open(PAIRED_PATH, 'w', encoding='utf-8') as f:
    json.dump(paired_data, f, ensure_ascii=False, indent=2)

## Control of short positive real reviews

In [ ]:
def is_short_positive_real(text):
    t = f' {normalize_text(text).lower()} '
    has_positive = any(w in t for w in POSITIVE_MARKERS)
    has_negative = any(w in t for w in NEGATIVE_MARKERS) or ' не рекомендую' in t or 'не советую' in t
    return len(text) < 90 and has_positive and not has_negative

short_positive_real = [t for t in real_wb_df['text'].tolist() if is_short_positive_real(t)]
for t in short_positive_real[:5]:
    display(f'[{len(t)}] {t}')


## Building dataset

In [ ]:
if 'paired_data' not in globals():
    with open(PAIRED_PATH, encoding='utf-8') as f:
        paired_data = json.load(f)

pairs_df = pd.DataFrame(paired_data)
required_cols = {'pair_id', 'real', 'fake'}
missing = required_cols - set(pairs_df.columns)
if missing:
    raise ValueError(f'В paired_reviews.json не хватает колонок: {missing}')

pairs_df['real'] = pairs_df['real'].map(normalize_text)
pairs_df['fake'] = pairs_df.apply(
    lambda row: trim_fake_to_real_length(row['real'], row['fake']),
    axis=1,
)
pairs_df = pairs_df.drop_duplicates(subset=['real', 'fake']).reset_index(drop=True)
pairs_df['orig_pair_id'] = pairs_df.get('pair_id', pd.Series([''] * len(pairs_df))).astype(str)
pairs_df['pair_id'] = [f'wb_pair_{i:05d}' for i in range(len(pairs_df))]


def balanced_quality_pair(real_text, fake_text):
    real_text = normalize_text(real_text)
    fake_text = normalize_text(fake_text)

    if fake_text == real_text:
        return False
    if len(fake_text) < 30:
        return False

    audit = pair_audit(real_text, fake_text)
    length_ok = 0.45 <= audit['len_ratio'] <= 1.10
    template_ok = audit['fake_template_count'] <= 3

    return length_ok and template_ok


audit_rows = []
for _, row in pairs_df.iterrows():
    audit = pair_audit(row['real'], row['fake'])
    audit['quality_ok'] = balanced_quality_pair(row['real'], row['fake'])
    audit_rows.append(audit)

audit_df = pd.DataFrame(audit_rows)
pairs_df = pd.concat([
    pairs_df.drop(columns=[c for c in audit_df.columns if c in pairs_df.columns], errors='ignore'),
    audit_df
], axis=1)

if 'prompt_version' in pairs_df.columns:
    pairs_df['prompt_version'] = pairs_df['prompt_version'].fillna('claude_paired').astype(str)
    pairs_df['prompt_version'] = pairs_df['prompt_version'].map(
        lambda x: x if 'len_balanced' in x else f'{x}_len_balanced'
    )
else:
    pairs_df['prompt_version'] = 'claude_paired_len_balanced'

filtered_pairs = pairs_df[pairs_df['quality_ok']].copy().reset_index(drop=True)
filtered_pairs = filtered_pairs.drop(columns=['rating', 'product_name', 'category'], errors='ignore')
display('Пар до фильтрации:', len(pairs_df))
display('Пар после фильтрации:', len(filtered_pairs))

meta_cols = [
    'pair_id', 'orig_pair_id', 'generator_model', 'prompt_version',
    'real_len', 'fake_len', 'len_ratio', 'real_bucket', 'fake_bucket'
]
meta_cols = [c for c in meta_cols if c in filtered_pairs.columns]

real_rows = filtered_pairs[meta_cols + ['real']].rename(columns={'real': 'text'})
real_rows['is_fake'] = 0
real_rows['source'] = 'wildberries_real'

fake_rows = filtered_pairs[meta_cols + ['fake']].rename(columns={'fake': 'text'})
fake_rows['is_fake'] = 1
fake_rows['source'] = 'synthetic_llm_tone_matched_len_balanced'

dataset_df = pd.concat([real_rows, fake_rows], ignore_index=True)
dataset_df['text'] = dataset_df['text'].map(normalize_text)

duplicate_pair_ids = dataset_df.loc[dataset_df['text'].duplicated(keep=False), 'pair_id'].unique()
if len(duplicate_pair_ids):
    dataset_df = dataset_df[~dataset_df['pair_id'].isin(duplicate_pair_ids)].copy()

pair_counts = dataset_df.groupby('pair_id')['is_fake'].nunique()
complete_pair_ids = pair_counts[pair_counts == 2].index
dataset_df = dataset_df[dataset_df['pair_id'].isin(complete_pair_ids)].reset_index(drop=True)
filtered_pairs = filtered_pairs[filtered_pairs['pair_id'].isin(complete_pair_ids)].reset_index(drop=True)

dataset_df = dataset_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

display('Итоговый датасет:', len(dataset_df), 'строк')
display(dataset_df['is_fake'].value_counts().rename({0: 'real', 1: 'fake'}))
display(dataset_df.assign(text_len=dataset_df['text'].str.len()).groupby('is_fake')['text_len'].describe().round(1))
display('len_ratio fake/real:')
display(filtered_pairs['len_ratio'].describe().round(3))
display(dataset_df.sample(min(6, len(dataset_df)), random_state=SEED)[['pair_id', 'text', 'is_fake', 'source']])

In [ ]:
DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
dataset_df.to_csv(DATA_DIR / 'e-commerce_dataset.csv', index=False, encoding='utf-8')
dataset_df.to_csv(DATA_DIR / 'paired_wb_fake_len_balanced.csv', index=False, encoding='utf-8')

fake_export = (
    dataset_df[dataset_df['is_fake'] == 1]
    .sort_values('pair_id')
    .to_dict(orient='records')
)
with open('fake_wb_final.json', 'w', encoding='utf-8') as f:
    json.dump(fake_export, f, ensure_ascii=False, indent=2)

filtered_pairs.to_json('paired_reviews_len_balanced.json', orient='records', force_ascii=False, indent=2)

